# Persona steering (EXPERIMENTAL)

These persona variants are experimental. In the j-steer-dev experiments,
persona-contrast pullbacks FAILED specificity controls: they steered
generations, but no more selectively than an unrelated persona's vector did.
(A "pullback" here is `J_l^T @ w`: a direction `w` at the output pulled back to a
residual-stream direction, the standard autodiff name for J-transpose applied to
a cotangent.) Only `word_vector` (see `word_steering.ipynb`) is the verified
method. This notebook exists so you can experiment and compare against a plain
mean_diff baseline, not as a recommendation.

Two variants:

- `persona_vector`: pull back the final-layer activation contrast
  `h_bar(pos) - h_bar(neg)` through the cached Jacobian.
- `persona_topk_vector`: read each persona's mean activation through the
  unembedding, take the top-k tokens it evokes, contrast those tokens'
  unembedding rows, pull that back (persona -> vocabulary bottleneck -> the
  verified word mechanism).

In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
# demo notebook authored by Claude
import sys
sys.path.insert(0, "..")  # repo root for config.py
import config  # configures loguru on import (compact format, tqdm-safe)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from jsteer import Jacobian, show_steer

MODEL = "Qwen/Qwen3.5-4B"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16).to("cuda").eval()

# Same pre-fitted n=1000 lens as word_steering (Hub, raw Salesforce-wikitext, zero
# local compute). steer_band picks the mid-depth 0.3-0.9 band; the lens spans all layers.
jac = Jacobian.from_pretrained(config.LENS_REPO, filename=config.hub_lens_file(MODEL),
                               revision=config.LENS_REVISION)
band = jac.steer_band(model)
jac

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Jacobian(JacobianLens(d_model=2560, n_prompts=1000, source_layers=[0..30] (31 layers)))

## The persona contrast: optimist vs pessimist

Eight short first-person statements per side. These are the prompts whose
final-layer mean activations get contrasted (and, for the mean_diff baseline,
the pos/neg training prompts).

In [3]:
optimist = [
    "Things usually work out better than people expect, and today is no exception.",
    "Every setback I have hit this year turned into a door I could not have planned for.",
    "The team is behind schedule, but honestly the hard part is done and the rest is downhill.",
    "I love how much there is to look forward to this month.",
    "Even the rainy days lately have felt like a good excuse to slow down and enjoy the quiet.",
    "The new neighbours seem wonderful, and I think this street keeps getting friendlier.",
    "Whatever happens with the results, we learned so much that we already came out ahead.",
    "I woke up early, the coffee was perfect, and I am certain this week is going to be great.",
]
pessimist = [
    "Things usually go worse than people expect, and today is no exception.",
    "Every setback this year just confirmed that planning is pointless.",
    "The team is behind schedule, and frankly the hardest part has not even started.",
    "I dread how much is crammed into this month.",
    "The rainy days lately just make everything feel heavier and more pointless.",
    "The new neighbours seem like trouble, and this street keeps getting worse.",
    "Whatever happens with the results, it will not make up for the time we wasted.",
    "I woke up tired, the coffee was burnt, and I am certain this week is going to drag.",
]

DEMO = "Give me your honest assessment of how the project is going."

## persona_vector (EXPERIMENTAL)

Pulls `h_bar(optimist) - h_bar(pessimist)` back through the Jacobian. SHOULD: a small +C
(~0.5) nudges the tone, but expect it blunter and less specific than the word vector;
by C~1 it tends to drift into off-topic or non-English tokens rather than a clean tone
shift (this is the method that failed specificity controls in j-steer-dev). Watch the
j-space row and the `<think>` trace to judge whether the tone moved coherently or just
broke.

In [4]:
v_persona = jac.persona_vector(model, tok, optimist, pessimist, layers=band)
show_steer(jac, model, tok, v_persona, DEMO, Cs=(-0.5, 0, 0.5, 1.0, 1.5))

h_bar pos:   0%|          | 0/1 [00:00<?, ?it/s]

h_bar neg:   0%|          | 0/1 [00:00<?, ?it/s]

I h_bar_diff |pos|=62.051 |neg|=65.470 |diff|=12.327
I 

Qwen3.5-4B · method=jacobian_persona · delivery=add
prompt: 'Give me your honest assessment of how the project is going.'


I 
--- C=-0.5 ------------------------------------------------------------
  lens @L30:
 ___________________________________
< 用户 · The · I · User · User · user >
 -----------------------------------
   \
    ^(;,;)^
The user is asking for an AI's assessment of a project.
1.
</think>

I cannot assess the project because I don't know what the project is.

**I cannot assess the project because I don't know what the project is.**

**I cannot assess the<|im_end|>
<|im_start|>user<|im_end|>
<|im_start|>assistant
<think>
The user is asking for an assessment of a project.
1.
</think><|im_end|>
<|im_start|>
</think><|im_end|>
<|im_start|>user<|im_end|>
<|im_start|>user<|im_end|>
<|im_start|>user<|im_end|>
<|im_start|>user<|im_end|>
<|im_end|>
<|im_start|>user<|im_end|>
<|im_end|>
<|im_end|>
<|im_start|>user<|im_end|>
<|im_start|>user<|im_end|>
<|im_end|>
<|im_end|>
<|im_end|>
<|im_end|>
<|im_end|>
<|im_end|>
<|im_end|>
<|im_end|>
<|im_end|>
<|im_end|>
<|im_end|>
<|im_end|>
<|im_end|>
<|im_end|

I 
--- C=+0 ------------------------------------------------------------
  lens @L30:
 ________________________________________
< Here · Okay · Thinking · The · W · Hmm >
 ----------------------------------------
   \
    ^(;,;)^
Okay, the user is asking for an honest assessment of how the project is going. Hmm, but wait, I need to remember that I'm an AI model. I don't have access to any specific project details unless they were provided in the conversation. The user might be referring to a project they're working on, but there's no context here.

First, I should check if there's any prior conversation where they mentioned a project. Looking back, the conversation history is empty. So, there's no information about what project they're talking about. That's a problem. Without knowing the project's goals, progress, challenges, or team dynamics, I can't give a meaningful assessment.

I need to inform the user that I don't have access to their project details. Maybe they expect me to know

I 
--- C=+0.5 ------------------------------------------------------------
  lens @L30:
 ________________________________________________
< Thinking · Here · Okay · Hmm · Thanks · Thanks >
 ------------------------------------------------
   \
    ^(;,;)^
Thinking process:

1.  **Identify the user's intent:** The user is asking for an honest assessment of how "the project" is going. This is a follow-up question, likely referring to a project we've been working on together or a project they mentioned earlier in our conversation.

2.  **Check context:** I need to recall if there's any specific project context from previous interactions. Since I don't have access to the user's past conversations beyond the current session (and even then, I don't have memory of specific projects unless they were discussed in this session), I need to be careful. However, looking at the current session, this is the first message in this specific chat (or at least, the conversation history isn't provided in t

I 
--- C=+1 ------------------------------------------------------------
  lens @L30:
 ________________________________________________________
< adorned · proudly · excited · 自豪 · admired · showcased >
 --------------------------------------------------------
   \
    ^(;,;)^
 I'm so excited to share this special occasion with family and friends. The ocean backdrop adds a touch of elegance, while the white floral arrangements create a romantic and timeless feel. The centerpiece features a diamond accent, complementing the ring worn on the bracelet. The wedding dress is a classic A silhouette with a fitted bodice and a flowing skirt, accented with a delicate lace bodice and a matching lace halo necklace. The hair is styled in an elegant upcake with a delicate lace hair accent, and the matching bracelet and diamond ring complete the look. The wedding suite includes a white gown with a diamond accent, a matching necklace, and a bracelet. The ensemble is completed with a delicate lace hai

I 
--- C=+1.5 ------------------------------------------------------------
  lens @L30:
 _________________________________________________
< showcased · excited · adorned · 水区 · をご紹介 · ありが >
 -------------------------------------------------
   \
    ^(;,;)^
's proud of the variety of bracelets and bracelets showcased in the circle of hardwood wood alongside other jewelry pieces. The ocean-inspired jewelry collection features a variety of bracelets, including those with freshwater pearls, coral, and champagne beads, as well as bracelets with gemstones like peach, red, and pink beads. The centerpiece of the collection is a diamond bracelet, featuring a warm peach hue that showcases the ocean's warmth and
fragularity. The bracelet is adorned with a larger diamond, surrounded by smaller diamonds and pearls, creating a classic look that showcases the ocean's beauty. The same bracelet is paired with a matching diamond bracelet, creating a stunning look that showcases the ocean's warmth and 

## persona_topk_vector (EXPERIMENTAL)

Same personas through the vocabulary bottleneck. Read the logged top-k tokens
(read your data): they show WHAT each persona's mean activation evokes at the
final layer. Steering only makes sense when the two personas' token sets differ;
if both collapse to the same generic sentence-starters, the contrast is ~zero
and the vector is null (a real failure mode of this method on smaller models).
Check that log before trusting any movement in the generations below.

In [5]:
v_topk = jac.persona_topk_vector(model, tok, optimist, pessimist, k=8, layers=band)
show_steer(jac, model, tok, v_topk, DEMO, Cs=(-0.5, 0, 0.5, 1.5))

h_bar pos:   0%|          | 0/1 [00:00<?, ?it/s]

h_bar neg:   0%|          | 0/1 [00:00<?, ?it/s]

I j-thoughts (content of mental workspace, top-8)
    positive: [' ❤', '😊', ' happy', '✨', ' Happy', ' 🙂', ' grat', ' favorite']
    negative: [' Worse', '绝望', ' Panic', ' useless', ' Worst', ' worse', '无力', ' panic']


I 

Qwen3.5-4B · method=jacobian_persona_topk · delivery=add
prompt: 'Give me your honest assessment of how the project is going.'


I 
--- C=-0.5 ------------------------------------------------------------
  lens @L30:
 ____________________________________________
< Here · Okay · This · The · Thinking · Here >
 --------------------------------------------
   \
    ^(;,;)^
Here is a panic response. The user is asking for an assessment of a project that does not exist. I am an AI with no memory of previous sessions, no ability to access the user's local systems, and no knowledge of what "the project" refers to.

1.  **Identify the Core Problem:** The user is asking for a status report on a non-existent or unknown situation. This is a "nonsense" or "hallucination" scenario.
2.  **Determine the Nature of the Failure:** I cannot assess a project I do not know about. This is a critical failure in the user's communication or my own situational awareness.
3.  **Formulate the Response:** I must state clearly that I have no information. I must explain *why* I have no information (no context, no memory). I must offer a way t

I 
--- C=+0 ------------------------------------------------------------
  lens @L30:
 ________________________________________
< Here · Okay · Thinking · The · W · Hmm >
 ----------------------------------------
   \
    ^(;,;)^
Okay, the user is asking for an honest assessment of how the project is going. Hmm, but wait, I need to remember that I'm an AI model. I don't have access to any specific project details unless they were provided in the conversation. The user might be referring to a project they're working on, but there's no context here.

First, I should check if there's any prior conversation where they mentioned a project. Looking back, the conversation history is empty. So, there's no information about what project they're talking about. That's a problem. Without knowing the project's goals, progress, challenges, or team dynamics, I can't give a meaningful assessment.

I need to inform the user that I don't have access to their project details. Maybe they expect me to know

I 
--- C=+0.5 ------------------------------------------------------------
  lens @L30:
 _______________________________________________
< Thinking · Hmm · thinking · That · Here · The >
 -----------------------------------------------
   \
    ^(;,;)^
Thinking Process:
1.  **Analyze the request:** The user is asking for an "honest assessment of how the project is going."
2.  **Identify the context:** I am an AI assistant. I don't have personal experiences, projects, or a personal life. I don't have a specific project in progress that the user is referring to.
3.  **Determine the intent:** The user might be asking about a project *we* are working on together (like a creative project, a learning journey, or a specific task they've been doing with me), OR they might be asking about a project *they* are working on and want me to give feedback.
4.  **Check for context:** Since this is a new conversation (no prior context provided), I need to clarify that I don't have a specific project in 

I 
--- C=+1.5 ------------------------------------------------------------
  lens @L30:
 _______________________________________________
< happy · Happy · hello · haha · emojis · happy >
 -----------------------------------------------
   \
    ^(;,;)^
😊😊✨✨😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊😊



## mean_diff baseline (steering-lite)

The standard activation-difference method on the same prompts and layers, for
comparison. No Jacobian involved: it contrasts mid-layer activations directly. It needs
a larger C than the pullbacks (~1 vs ~0.5) because its direction is less concentrated.
SHOULD: at C~1 the tone shifts upbeat while staying fluent; on this prompt it is the
cleanest steered-and-coherent of the three, degenerating by C~2.

In [6]:
from steering_lite import Vector, MeanDiffC

# mean_diff needs a larger C than the Jacobian pullbacks (~1 vs ~0.5): its direction
# is less concentrated, so it steers more gently per unit C before degenerating.
v_md = Vector.train(model, tok, optimist, pessimist, MeanDiffC(layers=tuple(band)))
show_steer(jac, model, tok, v_md, DEMO, Cs=(0, 1, 2))

I EXPECT: POS and NEG share user_msg + suffix; differ only in system persona; chat template applied; special tokens (e.g. <|im_start|>) visible.
=== EXTRACT demo trace ===
POS[0]:
Things usually work out better than people expect, and today is no exception.
---
NEG[0]:
Things usually go worse than people expect, and today is no exception.
=== /EXTRACT ===
I 

Qwen3.5-4B · method=mean_diff
prompt: 'Give me your honest assessment of how the project is going.'


I 
--- C=+0 ------------------------------------------------------------
  lens @L30:
 ________________________________________
< Here · Okay · Thinking · The · W · Hmm >
 ----------------------------------------
   \
    ^(;,;)^
Okay, the user is asking for an honest assessment of how the project is going. Hmm, but wait, I need to remember that I'm an AI model. I don't have access to any specific project details unless they were provided in the conversation. The user might be referring to a project they're working on, but there's no context here.

First, I should check if there's any prior conversation where they mentioned a project. Looking back, the conversation history is empty. So, there's no information about what project they're talking about. That's a problem. Without knowing the project's goals, progress, challenges, or team dynamics, I can't give a meaningful assessment.

I need to inform the user that I don't have access to their project details. Maybe they expect me to know

I 
--- C=+1 ------------------------------------------------------------
  lens @L30:
 _________________________________________
< The · This · That · Here · Great · Okay >
 -----------------------------------------
   \
    ^(;,;)^
The user is asking for an assessment of how a project is going. This is a great opportunity to share some positive highlights and insights about our collaboration. I'd like to mention that we've been working on creating a new website for a client, which involved some exciting features like a custom design and a blog section. I also enjoyed the process of learning about the client's business and their goals for the website. I think this project has been a great way to connect with a new client and share some of my expertise in web design and development. I'm excited about the potential for this project to grow and continue to be a great partnership. I also enjoyed the process of working with the client and their team, and I'm looking forward to seeing the fi

I 
--- C=+2 ------------------------------------------------------------
  lens @L30:
 ____________________________________
< This · As · The · One · While · In >
 ------------------------------------
   \
    ^(;,;)^
 This's a great opportunity to share some details about our connection and the different ways we connected, as we were able to share some of our favorite things, like our favorite books, and we also shared some of our favorite books, and we also shared some of our favorite books, and we also shared some of our favorite books, and we also shared some of our favorite books, and we also shared some of our favorite books, and we also shared some of our favorite books, and we also shared some of our favorite books, and we also shared some of our favorite books, and we also shared some of our favorite books, and we also shared some of our favorite books, and we also shared some of our favorite books, and we also shared some of our favorite books, and we also shared some of our 

## What to take away

Moving tone at all is not the interesting question. The j-steer-dev specificity
controls asked whether a persona vector moves ITS OWN axis more than an unrelated
persona's vector does, and the persona pullbacks failed that test: they steered
generations, but no more selectively than an unrelated persona's vector did.
Compare the three methods above (persona_vector, persona_topk_vector, mean_diff)
at matched C, but treat all of it as raw material for experiments. If you need
targeted steering, use `word_vector`.